In [ ]:
import pandas as pd
import numpy as np

ruta=r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\SLA_Gold.parquet"
sla = pd.read_parquet(ruta)

# Asegurar que la columna de fecha sea datetime (evita comparaciones con strings)
sla["Fecha Finalizacion"] = pd.to_datetime(sla["Fecha Finalizacion"], errors="coerce")

decem = sla[sla["Fecha Finalizacion"] >= pd.Timestamp('2026-01-01')]

# Calcular medias solo para columnas numéricas y de tipo timedeltas (SLA)
num_mean = decem.select_dtypes(include=[np.number]).mean()
td_mean = decem.select_dtypes(include=['timedelta64[ns]']).mean()
mean_sla = pd.concat([num_mean, td_mean])

print("Mean SLA:")
print(mean_sla)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definir tu Top 5 basado en el análisis de Pareto
top_5_soluciones = [
    'CONECTOR DAÑADO',
    'CAMBIO DE VLAN',
    'CONFIGURACION DE EQUIPO',
    'FALLA GENERAL',
    'FALLA LOS'
]

# 2. Filtrar el DataFrame limpio (df_final) para quedarnos solo con el Top 5
df_pareto = df_final[df_final['Solucion Aplicada'].isin(top_5_soluciones)].copy()

# 3. Graficar los histogramas separados para ver la "limpieza" de las curvas
g = sns.FacetGrid(df_pareto, col="Solucion Aplicada", col_wrap=3, height=4, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="SLA Resolucion Min", kde=True, bins=30, color='teal')

# Dar formato a los gráficos
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Minutos de Resolución", "Cantidad de Tickets")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Supongamos que df_pareto es tu tabla ya filtrada con el Top 5 de soluciones
# 1. Definir el tamaño del "depósito" o bin (ej. cada 60 minutos = 1 hora)
ancho_bin = 60
max_minutos = int(df_pareto['SLA Resolucion Min'].max())
bins = range(0, max_minutos + ancho_bin, ancho_bin)

# 2. Crear las etiquetas (ej. "0-59", "60-119", etc.)
etiquetas = [f"{i} a {i+ancho_bin-1}" for i in bins[:-1]]

# 3. Asignar cada ticket a su rango correspondiente
df_pareto['Rango_Tiempo_Min'] = pd.cut(df_pareto['SLA Resolucion Min'], bins=bins, labels=etiquetas, right=False)

# 4. AGRUPAR: Esta es la magia que comprime los datos para Power BI
df_gold_distribucion = df_pareto.groupby(['Solucion Aplicada', 'Rango_Tiempo_Min']).size().reset_index(name='Cantidad_Tickets')

# Limpiar rangos donde no hubo ningún ticket para ahorrar aún más peso
df_gold_distribucion = df_gold_distribucion[df_gold_distribucion['Cantidad_Tickets'] > 0]
display(df_gold_distribucion)
# Exportar a Parquet
# df_gold_distribucion.to_parquet("Gold_Distribucion_SLA.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet')
display(df)

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
print(df.columns)
df = df.groupby(['Hora de Pago', 'Vendedor'], as_index=False).size()


display(df)

In [ ]:
import polars as pl
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Downloads\Ventas_Estatus_Gold-Sharepoint.parquet')

mask = (df['Fecha']>=('2026-02-01')) & (df['Fecha']<=('2026-02-28')) &(df['Vendedor'].str.contains("OFI", case=False, na=False))
df_feb=df[mask]
print(df_feb.columns)
df_feb = df_feb.drop_duplicates(subset=['N° Abonado', 'Documento','Fecha','Vendedor','Ciudad','Hora'])
display(df_feb.describe(include='all'))
display(df_feb)

In [ ]:
import pandas as pd
import polars as pl
from pathlib import Path
import duckdb

rutahome = Path.home()
ruta_base = rutahome / 'Documents' / 'A-DataStack' / '01-Proyectos' / '01-Data_PipelinesFibex' / '02_Data_Lake' / 'gold_data'

lf = duckdb.query


In [ ]:
import pandas as pd 

# Carga de datos
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Estatus_Gold.parquet')
df_ventas = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet')
df_afluencia = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Afluencia_Gold.parquet')

# --- CORRECCIONES DE FILTRADO ---

# 1. Filtrar df_fl (Afluencia) - Marzo y Ventas
df_fl = df_afluencia[(df_afluencia['Fecha'].dt.month == 3) & (df_afluencia['Tipo de afluencia'] == 'Ventas')]

# 2. Filtrar df (Estatus) - Usando su propia columna de fecha (asumiendo que se llama 'Fecha')
# Si la columna en 'df' tiene otro nombre, cámbiala aquí:
df = df[df['Fecha'].dt.month == 3] 

# 3. Filtrar df_ventas (Listado) - Marzo y Canal específico
df_ventas = df_ventas[(df_ventas['Fecha Contrato'].dt.month == 3) & (df_ventas['Canal'] == 'OFICINA COMERCIAL')]

# --- CORRECCIONES DE SALIDA ---

# Agrupación y conteo (se eliminó .reindex para que se muestren los datos)
print("Resumen de Estatus (df):")
print(df.groupby(['Estatus'], as_index=False).agg(Cantidad = ('N° Abonado', 'size')))

print("\nResumen de Estatus (df_fl):")
print(df_fl.groupby(['Estatus'], as_index=False).agg(Cantidad = ('N° Abonado', 'size')))

print(f"\nTotal (df): {df.count()['N° Abonado']}")
print(f"Total (df_fl): {df_fl.count()['N° Abonado']}")

In [ ]:
import pandas as pd 
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Ventas_Listado_Bronze.parquet')
display(df.columns)

In [ ]:
import pandas as pd


df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Horas_Raw_Bronze.parquet')
df_recaudacion = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Recaudacion_Raw_Bronze.parquet')
display(df.columns)
display(df)
display(df_recaudacion.columns)
display(df_recaudacion)
df_merge = pd.merge(df_recaudacion, df, on='ID Pago', how='inner')
display(df_merge.head())

In [2]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
display(df)

,ID Contrato,N° Abonado,Fecha,Total Pago,Forma de Pago,Banco,Oficina,Fecha Contrato,Estatus,Suscripción,Grupo Afinidad,Nombre Franquicia,Ciudad,Vendedor,Tipo de afluencia,Clasificacion,Hora de Pago
0,'CONT21FE5D5030705988',LCH58251,2025-07-01,16.64,TARJETA DE DEBITO,PROVINCIAL,OFC COMERCIAL CUMANA,2025-05-12 00:00:00,ACTIVO,30,HOGAR,FIBEX ANZOATEGUI,CUMANA,JORGE PAREJO OFIC CUMANA,RECAUDACIÓN,OFICINAS PROPIAS,07:00
1,'CONT82B7255301054845',GU17461,2025-07-01,30.00,TARJETA DE DEBITO,100% BANCO TH,OFC - VALLE LA PASCUA,2024-07-01 00:00:00,ACTIVO,30,HOGAR,FIBEX GUARICO,VALLE DE LA PASCUA,GERONIMO A ZAMBRANO ASESOR OFICINA GUARICO,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
2,'CONT3CA2F6D7A1834761',PTO10344,2025-07-01,30.00,TARJETA DE DEBITO,PROVINCIAL,OFI-PTO CABELLO,2024-11-19 00:00:00,ACTIVO,30,HOGAR,FIBEX PTO CABELLO,PUERTO CABELLO,CARILYN MELENDEZ OFIC PUERTO CABELLO,RECAUDACIÓN,OFICINAS PROPIAS,08:00
3,'CONT9E17A54A36F85147',LV2565,2025-07-01,20.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-02-01 00:00:00,ACTIVO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
4,'CONT0F889188DCA55438',LV3444,2025-07-01,10.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-04-28 00:00:00,CORTADO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1537689,'CONT6625705F0D043154',FM1739,2026-03-06,35.00,TARJETA DE DEBITO,PROVINCIAL,OFC COBRO CENTRO,2023-11-28 00:00:00,ACTIVO,35,HOGAR,FIBEX MATURIN,MATURIN,ELOINA ARTEAGA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
1537690,'CONTC07586C5D9929991',V9476,2026-03-06,39.99,TARJETA DE DEBITO,100% BANCO,OFI-METROPOLIS,2022-07-02 00:00:00,ACTIVO,42,HOGAR,FIBEX VALENCIA,VALENCIA,MAYERLIN GARCIA WALTER OFIC METROPOLIS VALENCIA,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537691,'CONTB747062EC6A45817',C2933,2026-03-06,40.00,EFECTIVO DOLLAR,,OFI-PARAISO,2023-01-05 00:00:00,ACTIVO,42,HOGAR,FIBEX CARACAS,CARACAS,KARIANNYS RODRIGUEZ OFIC EL PARAISO CARACAS,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537692,'CONTA384F1D4C4149627',V122291,2026-03-06,25.00,TARJETA DE DEBITO,PROVINCIAL,OFC - GUACARA,2025-08-18 00:00:00,ACTIVO,25,HOGAR,FIBEX VALENCIA,VALENCIA,YELIUSKAR MELENDEZ AGENTE ESCALA 23 GUACARA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
